# ITS — Baseline Score-Based Sampler

Uses the `src/its.*` API.

## Contents
1. Load score-model checkpoint
2. Baseline SDE sampling
3. DDPM / DDIM sampling
4. Visualise sample grids
5. Quick FID/IS evaluation

In [ ]:
import sys
from pathlib import Path

ROOT = Path('.').resolve().parent
if str(ROOT / 'src') not in sys.path:
    sys.path.insert(0, str(ROOT / 'src'))
print('Project root:', ROOT)

In [ ]:
import torch
from its.models import ScoreUNetConfig, build_score_model
from its.sde import ScoreSDEConfig, ScoreSDESimulator

DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Device:', DEVICE)

## 1. Load checkpoint

In [ ]:
CKPT_DIR = ROOT / 'checkpoints' / 'score'
checkpoints = sorted(CKPT_DIR.glob('score_epoch_*.pt'))
CKPT_PATH = checkpoints[-1] if checkpoints else None
print('Latest checkpoint:', CKPT_PATH)

In [ ]:
if CKPT_PATH:
    state = torch.load(CKPT_PATH, map_location='cpu')
    if 'model_config' in state:
        model_cfg = ScoreUNetConfig(**state['model_config'])
        print('Config from checkpoint:', model_cfg)
    else:
        # Fallback for the epoch-50 checkpoint trained with base_channels=128.
        model_cfg = ScoreUNetConfig(in_channels=3, base_channels=128, channel_mults=(1, 2, 2, 4))
        print('WARNING: using fallback config (no model_config in checkpoint)')
    score_model = build_score_model(model_cfg)
    score_model.load_state_dict(state['model'])
    score_model.to(DEVICE).eval()
    print('Parameters:', sum(p.numel() for p in score_model.parameters()))
else:
    model_cfg = ScoreUNetConfig(in_channels=3, base_channels=32)
    score_model = build_score_model(model_cfg).to(DEVICE).eval()
    print('No checkpoint — using random weights.')

## 2. Baseline SDE sampling

In [ ]:
sde_cfg = ScoreSDEConfig(beta_min=0.1, beta_max=20.0, num_steps=200,
                        sigma_min=0.01, sigma_max=1.0, control_weight=0.0)
simulator = ScoreSDESimulator(score_model, sde_cfg)
BATCH = 16
with torch.no_grad():
    samples, stats = simulator.sample(
        shape=(BATCH, model_cfg.in_channels, 32, 32),
        device=torch.device(DEVICE), return_stats=True,
    )
print('Shape:', samples.shape, '| Stats:', stats)

In [ ]:
import matplotlib.pyplot as plt
from torchvision.utils import make_grid

grid = make_grid(samples.cpu().clamp(-1, 1) * 0.5 + 0.5, nrow=4)
plt.figure(figsize=(8, 8))
plt.imshow(grid.permute(1, 2, 0).numpy())
plt.axis('off')
plt.title('Baseline SDE samples')
plt.show()

## 3. DDPM / DDIM sampling

The `DDPMSampler` internally calls `score_to_eps(score, sigma)` before the update rule.

In [ ]:
from its.samplers.ddpm import DDPMSampler, DDPMSamplerConfig

ddpm_cfg = DDPMSamplerConfig(num_steps=100, use_ddim=False)
ddpm = DDPMSampler(score_model, ddpm_cfg)
with torch.no_grad():
    ddpm_samples, ddpm_stats = ddpm.sample((BATCH, model_cfg.in_channels, 32, 32), torch.device(DEVICE))
print('DDPM NFE:', ddpm_stats['nfe_total'])

In [ ]:
ddim_cfg = DDPMSamplerConfig(num_steps=50, use_ddim=True, ddim_eta=0.0)
ddim = DDPMSampler(score_model, ddim_cfg)
with torch.no_grad():
    ddim_samples, ddim_stats = ddim.sample((BATCH, model_cfg.in_channels, 32, 32), torch.device(DEVICE))
print('DDIM NFE:', ddim_stats['nfe_total'])

## 4. Quick FID/IS evaluation (256 samples)

In [ ]:
from its.eval import EvaluationConfig, evaluate_sampler
from its.eval.evaluator import evaluate_ddpm_baseline

eval_cfg = EvaluationConfig(dataset_name='cifar10', num_samples=256, batch_size=64, device=DEVICE)
print('Evaluating baseline SDE ...')
results = evaluate_sampler(score_model, None, sde_cfg, eval_cfg)
print('Baseline SDE:', results)

In [ ]:
print('Evaluating DDPM ...')
ddpm_results = evaluate_ddpm_baseline(score_model, sde_cfg, eval_cfg, baseline='ddpm')
print('DDPM:', ddpm_results)